# medgraph review notebook

A guided check of the work so far. It runs the real pipeline on **synthetic Synthea data**:

1. **Phase 0.** The cohorts are pinned, intact and reproducible.
2. **Phase 1a.** One patient's FHIR bundle is read into the typed record model, and any value can be traced back to the resource it came from.
3. **Phase 1b.** That patient's labs are normalized into canonical units, and the parsers refuse to guess on ambiguous input.
4. **Cohort-wide.** Normalization outcomes and data problems across many patients.
5. **Tests.** The unit test suite, run from here.
6. **What to review** at this checkpoint.

**How to run.**
- Run `make setup` once (it installs the notebook kernel).
- Open this notebook in VS Code, pick the `.venv` kernel (Python 3.12), then **Run All**. It takes about a minute, and the T7 must be mounted.
- From a terminal, `make review` re-runs it in place.

**Privacy.** Use this notebook on synthetic data only. Outputs are stripped on commit (the nbstripout hook), but real records or MIMIC must never be opened here.

In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from medgraph.ingest.fhir import BundleError, read_bundle_file
from medgraph.ingest.files import content_digest, iter_data_files, sha256_file
from medgraph.normalize.analytes import ANALYTES, BY_KEY
from medgraph.normalize.labs import normalize_labs
from medgraph.normalize.numbers import parse_number
from medgraph.normalize.ranges import parse_reference_range
from medgraph.normalize.time import parse_fhir_time, parse_report_date
from medgraph.normalize.units import to_ucum
from medgraph.settings import Settings

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
settings = Settings(_env_file=REPO / ".env")
COHORTS = settings.synthea_dir
assert COHORTS.is_dir(), f"{COHORTS} not found: is the T7 mounted?"
DEV = COHORTS / "dev-1000"
LOINC = "http://loinc.org"

pd.set_option("display.max_colwidth", 90)
# Chart style: validated reference palette, recessive grid and axes, text in ink colors.
BLUE, ORANGE, CRITICAL = "#2a78d6", "#eb6834", "#d03b3b"
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "font.size": 10,
        "lines.linewidth": 2,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)
print(f"data folder: {settings.data_dir}")

## 1. Phase 0: pinned, intact, reproducible cohorts

`scripts/generate_synthea.py` writes a `MANIFEST.json` into every cohort folder. It records the exact command, the Synthea release (checked by SHA-256), the seeds, the fixed simulation dates, the JVM timezone, and a hash of every output file.

In [ ]:
rows = []
for manifest_path in sorted(COHORTS.glob("*/MANIFEST.json")):
    m = json.loads(manifest_path.read_text())
    cmd, out = m["command"], m["outputs"]
    rows.append(
        {
            "cohort folder": manifest_path.parent.name,
            "Synthea": m["config"]["synthea"]["version"],
            "seed / clinician seed": f"{m['config']['seed']} / {m['config']['clinician_seed']}",
            "-r / -e dates": f"{cmd[cmd.index('-r') + 1]} / {cmd[cmd.index('-e') + 1]}",
            "JVM timezone": next(a.split("=")[1] for a in cmd if a.startswith("-Duser.timezone")),
            "patients (deceased)": f"{out['n_patients']:,} ({out['n_deceased']})",
            "size (MB)": round(out["total_bytes"] / 1e6),
            "seconds": m["duration_s"],
            "content digest": out["content_digest"][:16] + "…",
        }
    )
pd.DataFrame(rows).set_index("cohort folder")

**How to read it.** `pilot-10` and `pilot-10-repeat` were generated separately; the second run pretended to be on a machine in New York (`TZ=America/New_York`). Identical content digests mean byte-identical output, so a run doesn't depend on when or where it happens.

Next, re-hash the files and compare them with the manifest. This detects corruption or accidental edits on the T7.

In [ ]:
VERIFY_DEV = False  # True also re-hashes all 4.2 GB of dev-1000 (about a minute)


def verify(cohort: Path) -> str:
    manifest = json.loads((cohort / "MANIFEST.json").read_text())["outputs"]
    fhir = cohort / "fhir"
    hashes = {
        p.relative_to(fhir).as_posix(): sha256_file(p) for p in iter_data_files(fhir, "*.json")
    }
    intact = hashes == manifest["files"] and content_digest(hashes) == manifest["content_digest"]
    return f"{cohort.name}: {len(hashes)} files, " + (
        "all match the manifest" if intact else "MISMATCH"
    )


for cohort in [COHORTS / "pilot-10", *([DEV] if VERIFY_DEV else [])]:
    print(verify(cohort))

The profile's findings (`docs/data/README.md`) are properties of the Synthea generator, and they shaped Phase 1:

In [ ]:
notes_md = (REPO / "docs/data/README.md").read_text()
display(Markdown(notes_md[notes_md.index("## What the dev-1000 profile shows") :]))

## 2. Phase 1a: one patient, from FHIR bundle to typed record

Pick the first dev-1000 patient with chronic kidney disease stage 3, and read their bundle with `ingest.fhir.read_bundle_file`.

In [ ]:
CKD3 = b'"433144002"'  # SNOMED CT: chronic kidney disease stage 3
bundle_path = next(p for p in iter_data_files(DEV / "fhir", "*.json") if CKD3 in p.read_bytes())
record = read_bundle_file(bundle_path)
patient = record.patient
born = patient.birth_date.date if patient.birth_date else "?"
died = patient.deceased.date if patient.deceased else "no"
print(f"record source: {record.source[:23]}…  (SHA-256 of the bundle file)")
print(f"patient {patient.id[:8]}…: {patient.gender}, born {born}, deceased: {died}")
print(
    "not read (out of scope):", ", ".join(f"{k} {v}" for k, v in record.skipped_resources.items())
)
print("ingestion issues:", dict(Counter(i.code for i in record.issues)) or "none")
pd.Series(
    {
        "encounters": len(record.encounters),
        "conditions": len(record.conditions),
        "medication requests": len(record.medication_requests),
        "observations": len(record.observations),
        "procedures": len(record.procedures),
        "diagnostic reports": len(record.diagnostic_reports),
        "notes (deduplicated)": len(record.notes),
    },
    name="records read",
).to_frame()

### Kidney-, anemia- and diabetes-related conditions

Notice how CKD stages pile up. Synthea doesn't close an earlier stage when it records the next one, so each condition's clinical status and abatement date are kept.

In [ ]:
TERMS = ("kidney", "renal", "anemia", "diabet", "albuminuria", "proteinuria")
pd.DataFrame(
    [
        {
            "onset": c.onset.date if c.onset else None,
            "condition": c.code.label,
            "SNOMED CT": c.code.code("http://snomed.info/sct"),
            "clinical status": c.clinical_status,
            "abated": c.abatement.date if c.abatement else "",
        }
        for c in record.conditions
        if any(t in c.code.label.lower() for t in TERMS)
    ]
).sort_values("onset", ignore_index=True)

### Every value is traceable

Take one creatinine observation. The typed record keeps a `SourceRef`. Following it back into the bundle finds the exact FHIR resource it came from, with the same value, unit and time. Note that the value keeps the digits exactly as written.

In [ ]:
obs = next(o for o in record.observations if o.code.code(LOINC) in ("38483-4", "2160-0"))
q = obs.value_quantity
print(f"typed record: {obs.code.label} = {q.value!r} {q.unit} at {obs.effective.instant}")
print(f"source ref:   {obs.source}")
raw = json.loads(bundle_path.read_bytes())
original = next(
    e["resource"] for e in raw["entry"] if e["resource"].get("id") == obs.source.resource_id
)
keep = ("resourceType", "id", "effectiveDateTime", "valueQuantity")
print("raw FHIR resource:", json.dumps({k: original[k] for k in keep}, indent=2), sep="\n")

### Notes

Synthea writes each clinical note twice, once as a DocumentReference and once as a DiagnosticReport. The parser merges identical text from the same encounter into one note that lists both sources.

In [ ]:
note = record.notes[-1]
print(
    f"{len(record.notes)} notes; the latest has sources:", [s.resource_type for s in note.sources]
)
print("-" * 60)
print(note.text[:900])

## 3. Phase 1b: the same patient's labs, normalized

`normalize.labs.normalize_labs` maps in-scope LOINC codes to the analyte registry and converts each value to the analyte's canonical unit. The original value and unit are kept alongside, and `status` says whether a rule may use the value: only `ok` values can.

In [ ]:
results = normalize_labs(record)
labs = pd.DataFrame(
    [
        {
            "date": r.effective.date if r.effective else None,
            "analyte": r.analyte,
            "method": r.method,
            "as recorded": (
                f"{r.original.value} {r.original.code or r.original.unit}"
                if r.original
                else r.qualitative
            ),
            "canonical": f"{r.value:.3g} {r.unit}" if r.value is not None else "",
            "status": r.status,
            "detail": r.detail or "",
        }
        for r in results
    ]
)
display(
    labs.pivot_table(
        index="analyte", columns="status", values="date", aggfunc="count", fill_value=0
    )
)
labs.sort_values("date").tail(12)

In [ ]:
def points(analyte: str, status: str) -> list[tuple]:
    return sorted(
        (r.effective.date, float(r.value))
        for r in results
        if r.analyte == analyte and r.status == status and r.value is not None and r.effective
    )


fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
panels = (
    ("creatinine", "Creatinine", "mg/dL"),
    ("egfr", "eGFR as reported (MDRD)", "mL/min/1.73 m²"),
)
for ax, (analyte, title, unit) in zip(axes, panels, strict=True):
    usable = points(analyte, "ok")
    if usable:
        # Dots, not a line: measurements are irregular and cluster during hospital stays.
        ax.plot(
            *zip(*usable, strict=True),
            linestyle="none",
            color=BLUE,
            marker="o",
            markersize=5,
            markeredgecolor=SURFACE,
            markeredgewidth=1,
        )
    excluded = points(analyte, "implausible")
    if excluded:
        ax.plot(
            *zip(*excluded, strict=True),
            linestyle="none",
            marker="X",
            markersize=7,
            color=CRITICAL,
            label="outside sanity bounds, not used",
        )
        ax.legend(loc="upper left")
    not_usable = sum(r.analyte == analyte and r.status != "ok" for r in results)
    note_text = f" · {not_usable} not usable" if not_usable else ""
    ax.set_title(f"{title}: {len(usable)} usable values{note_text}", loc="left", fontsize=10)
    ax.set_ylabel(unit)
    ax.set_ylim(bottom=0)
fig.suptitle(f"Lab history of patient {patient.id[:8]}… (synthetic)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

A chart only plots usable values. eGFR recorded in `mL/min` has a different unit, so it isn't drawn on this axis; it's counted as "not usable" in the panel title.

### The analyte registry

The analyte registry defines what's in scope. Each non-trivial conversion factor cites its source, and a test enforces this. The sanity bounds are deliberately wide engineering limits for catching unit mix-ups and corrupt values. They are **not** clinical thresholds.

In [ ]:
pd.DataFrame(
    [
        {
            "analyte": a.key,
            "LOINC codes": ", ".join(a.loinc),
            "canonical unit": a.canonical_unit,
            "also converts from": ", ".join(u for u in a.to_canonical if u != a.canonical_unit)
            or "none",
            "sanity bounds": f"{a.sanity_low} to {a.sanity_high}",
            "conversion source": a.conversion_source or "unit arithmetic",
        }
        for a in ANALYTES
    ]
).set_index("analyte")

### Edge cases: the parsers refuse to guess

Edit the list and re-run the cell to try your own inputs. A rejection names its reason; it is never silently interpreted.

In [ ]:
FUNCTIONS = {
    f.__name__: f
    for f in (parse_number, parse_report_date, parse_fhir_time, to_ucum, parse_reference_range)
}
CASES = [
    ("parse_number", "1,2", "it"),
    ("parse_number", "1.234", "it"),
    ("parse_number", "7.5", "it"),
    ("parse_number", "1,234", "en"),
    ("parse_number", "1,234", None),
    ("parse_number", "< 0,5", "it"),
    ("parse_report_date", "07/03/2025", "it"),
    ("parse_report_date", "07/03/2025", "en-US"),
    ("parse_report_date", "7 marzo 2025", "it"),
    ("parse_report_date", "07/03/25", "it"),
    ("parse_fhir_time", "2025-03"),
    ("parse_fhir_time", "2025-03-07T10:20:30"),
    ("to_ucum", "µmol/L"),
    ("to_ucum", "ml/min/1,73 mq"),
    ("to_ucum", "{presence}"),
    ("parse_reference_range", "0,70 - 1,20", "it"),
    ("parse_reference_range", "M: 13-17 F: 12-15", "en"),
]


def attempt(name: str, *args: object) -> str:
    try:
        result = FUNCTIONS[name](*args)
    except ValueError as exc:
        return f"rejected: {exc}"
    return "not recognised" if result is None else str(result)


pd.DataFrame(
    [
        {"function": name, "input": ", ".join(map(repr, args)), "result": attempt(name, *args)}
        for name, *args in CASES
    ]
)

## 4. Cohort-wide: normalization outcomes and data problems

Read `N_PATIENTS` patients through the same pipeline. Set it to `None` for all 1,148 (about a minute). `make normalization` writes the full-cohort version to `docs/data/synthea-dev-1000-normalization.md`.

In [ ]:
N_PATIENTS = 300

rows, issues, days_after_death = [], Counter(), []
n = 0
for path in iter_data_files(DEV / "fhir", "*.json"):
    if N_PATIENTS is not None and n >= N_PATIENTS:
        break
    try:
        rec = read_bundle_file(path)
    except BundleError:
        continue  # hospital and practitioner bundles
    n += 1
    issues.update(i.code for i in rec.issues)
    if rec.patient.deceased:
        death = rec.patient.deceased.date
        days_after_death += [
            (e.period.start.date - death).days
            for e in rec.encounters
            if e.period.start and e.period.start.date > death
        ]
    for r in normalize_labs(rec):
        rows.append(
            {
                "analyte": r.analyte,
                "status": r.status,
                "value": float(r.value) if r.value is not None else np.nan,
                "recorded unit": (r.original.code or r.original.unit) if r.original else None,
                "recorded value": float(r.original.value) if r.original else np.nan,
            }
        )
cohort = pd.DataFrame(rows)
print(f"{n:,} patients, {len(cohort):,} in-scope lab results")
pd.crosstab(cohort["analyte"], cohort["status"], margins=True, margins_name="total")

### Why some values are set aside

- **Left:** creatinine in canonical mg/dL, on a log scale. Values above the sanity bound are kept and shown but never used by rules.
- **Right:** eGFR as recorded. Synthea uses one LOINC code with two units, and the two sets of values clearly differ. The `mL/min` values aren't adjusted to body surface area, so they're kept separate from `mL/min/1.73 m²` instead of being merged.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

creatinine = cohort[cohort.analyte == "creatinine"]
bins = np.logspace(np.log10(0.2), np.log10(100), 40)
usable = creatinine.loc[creatinine.status == "ok", "value"]
excluded = creatinine.loc[creatinine.status == "implausible", "value"]
left.hist(
    usable,
    bins=bins,
    color=BLUE,
    edgecolor=SURFACE,
    linewidth=0.6,
    label=f"usable ({len(usable):,})",
)
if len(excluded):  # too few to see as bars: draw each one as a tick on the baseline
    left.scatter(
        excluded,
        [0.03] * len(excluded),
        transform=left.get_xaxis_transform(),
        marker="|",
        s=150,
        linewidths=2,
        color=CRITICAL,
        label=f"outside sanity bounds, not used ({len(excluded):,})",
    )
bound = float(BY_KEY["creatinine"].sanity_high)
left.axvline(bound, color=INK_2, linewidth=1, linestyle="--")
left.annotate(
    f"sanity bound {bound:g}",
    (bound, 0.97),
    xycoords=("data", "axes fraction"),
    xytext=(-4, 0),
    textcoords="offset points",
    ha="right",
    va="top",
    fontsize=8,
    color=INK_2,
)
left.set_xscale("log")
left.set_xlabel("creatinine, mg/dL (log scale)")
left.set_ylabel("lab results")
left.set_title("Creatinine", loc="left")
left.legend(loc="upper left")

egfr = cohort[cohort.analyte == "egfr"]
egfr_bins = np.linspace(0, 170, 35)
for unit, color, label in (
    ("mL/min/{1.73_m2}", BLUE, "mL/min/1.73 m², usable"),
    ("mL/min", ORANGE, "mL/min, not convertible"),
):
    values = egfr.loc[egfr["recorded unit"] == unit, "recorded value"]
    right.hist(
        values,
        bins=egfr_bins,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"{label} ({len(values):,})",
    )
right.set_xlabel("eGFR value as recorded")
right.set_title("eGFR (LOINC 33914-3) by recorded unit", loc="left")
right.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
print("ingestion issues:", dict(issues) or "none")
pd.Series(days_after_death, name="days from death to the later encounter").describe().to_frame()

## 5. Test suite

This runs the unit tests (about a second). In a terminal, `make test-synthea` also parses all of dev-1000 and checks that nothing in scope is dropped.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout[-700:] or result.stderr[-700:])

## 6. What to review at this checkpoint

- **Analyte scope and sanity bounds** (`src/medgraph/normalize/analytes.py`). The bounds are engineering judgement, wide on purpose. For example, creatinine counts as usable up to 40 mg/dL.
- **MedicationAdministration isn't read** (17k in-hospital administrations in dev-1000). Monitoring rules will work from prescriptions.
- **Sex-specific rules** (CKD-EPI, WHO hemoglobin thresholds) will use FHIR administrative gender as a proxy, and will say so.
- **Decisions and evidence:** `docs/decisions/0001-…` and `0002-…`.